# 🏦 Zarya Digital Bank — Day 4 Task 1: LoRA Adapter Merge & GGUF Quantization

This notebook merges the DPO-aligned adapter weights into `Qwen/Qwen2.5-3B-Instruct` and compiles:
1. **FP16 Merged Model** (Full Precision reference)
2. **GGUF Q4_K_M** (High-performance 4-bit for llama.cpp / vLLM / Ollama production serving)

Run this in Google Colab (with T4 or A100 GPU) or on your local RTX 2080 Ti GPU machine.

In [ ]:
# Step 1: Install Unsloth and llama.cpp export tools
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets transformers huggingface_hub


In [ ]:
# Step 2: Load Base Model and Trained DPO LoRA Adapter
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # Auto detection (Float16 / Bfloat16)
load_in_4bit = False  # Load in 16-bit for lossless weight merging

# Point to extracted DPO checkpoint
adapter_path = "./zarya_qwen2_5_3b_dpo"

print("Loading base model and DPO adapter weights...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = adapter_path,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print("Base model and DPO adapter successfully loaded!")


In [ ]:
# Step 3: Export Merged FP16 Standalone Model
print("Merging LoRA weights (W = W0 + alpha/r * BA) into FP16 standalone model...")
model.save_pretrained_merged("zarya_qwen2_5_3b_merged_fp16", tokenizer, save_method="merged_16bit")
print("✅ Merged FP16 model saved to ./zarya_qwen2_5_3b_merged_fp16")


In [ ]:
# Step 4: Quantize to 4-bit GGUF (Q4_K_M) for llama.cpp / Ollama
print("Quantizing merged weights to 4-bit GGUF (Q4_K_M)...")
model.save_pretrained_gguf("zarya_qwen2_5_3b_gguf", tokenizer, quantization_method="q4_k_m")
print("🎉 GGUF 4-bit model generated at ./zarya_qwen2_5_3b_gguf/unsloth.Q4_K_M.gguf")


In [ ]:
# Step 5: Test Model Verification Inference
FastLanguageModel.for_inference(model)
prompt = """<|im_start|>system
You are Zarya Bank's AI customer support assistant. Follow bank tone strictly.<|im_end|>
<|im_start|>user
Mera ATM card block karna hai gum ho gaya hai.<|im_end|>
<|im_start|>assistant
"""
inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True)
response = tokenizer.batch_decode(outputs)[0]
print("Model Response:")
print(response)


In [ ]:
# Step 6: (Optional) Push Quantized GGUF / Merged Model to Hugging Face Hub
from google.colab import userdata
import os

# HF_TOKEN = userdata.get('HF_TOKEN')
# model.push_to_hub_gguf("sairafatima193/zarya-qwen2.5-3b-gguf-v1", tokenizer, quantization_method="q4_k_m", token=HF_TOKEN)
print("Ready for Hugging Face upload whenever needed.")
